# 05 — Track 2: Sequence Model (GRU)

**What this notebook is for.** Track 1 (flat table + CatBoost) plateaued at a
validated **AUROC 0.8032 ± 0.0011**. The flat table compresses each lab's
day-by-day trajectory into a few summaries. Labs are the only block that changes
over time, so their *trajectory shape* — improving vs. worsening toward discharge
— is information the flat model can't see. A GRU reads the day-by-day sequence
directly and may capture it.

**The bar.** This model must beat **0.8032** on the same patient-grouped CV to be
worth its extra complexity. A negative result is still informative (it would tell
us lab trajectory shape carries little signal).

**Update (v2 — lab-focused).** A first version fed all 171 columns through the
GRU and scored 0.790, below the Track-1 bar. But 158 of those columns (demographics, ICD, meds) don't change across days — feeding constants through a recurrent layer dilutes the ~36 lab columns that actually vary. This version splits the input: the **lab sequence** (the part that varies) goes through the GRU, while **static features** (constant per admission) bypass the GRU and join at the end. This is the fair test of whether lab *trajectory shape* helps.

**Why GRU.** Fewer parameters than an LSTM (better for our small ~11k dataset);
simpler and more data-efficient than a Transformer (attention needs more data).

**Hardware.** This notebook uses the GPU if available and falls back to CPU
otherwise. It prints which one it found — check that line before a long run.

**How to read this notebook.** Every code cell has a plain-language *"How to read
this"* note.

## 0. Check PyTorch and the GPU

Run this first. If it prints `device: cuda`, the GPU is active. If it prints
`device: cpu`, training will still work but be much slower — stop and check your
CUDA/PyTorch install before running the whole notebook.

In [1]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

torch version: 2.9.1+cu128
cuda available: True
device: cuda
gpu: NVIDIA RTX A2000 Laptop GPU


**How to read this:** confirms the deep-learning library loaded and whether the
GPU is visible. **Remember:** if this says `cpu` and you expected `cuda`, the rest
of the notebook will be slow — fix the environment before continuing rather than
waiting on CPU training.

## 1. Load the raw sequences and labels

Unlike Track 1, we do **not** use the flat feature table — we go back to the raw
`feat_dict` (the day-by-day arrays), because the whole point is to feed the
sequence to the model. We still use the same admissions (train + valid) and the
same labels and patient groups, so the comparison to Track 1 is fair.

In [2]:
import os, pickle, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = "data"
EHR_PICKLE = os.path.join(DATA_DIR, "ehr_preprocessed_seq_by_day_cat_embedding.pkl")
LABEL_COL = "readmitted_within_30days"

with open(EHR_PICKLE, "rb") as f:
    ehr = pickle.load(f)
feat_dict = ehr["feat_dict"]

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid.csv"))

def to_binary(s):
    return s.map(lambda v: 1 if v in {1, "1", True, "True"} else 0).astype(int)
for df in (train_df, valid_df):
    df["y"] = to_binary(df[LABEL_COL])

adm = pd.concat([train_df, valid_df], ignore_index=True).drop_duplicates("id")
adm["subject_id"] = adm["id"].str.split("_").str[0]
adm = adm[["id", "subject_id", "y"]].reset_index(drop=True)

ids = adm["id"].tolist()
y = adm["y"].values.astype(np.float32)
groups = adm["subject_id"].values
print("admissions:", len(ids), "| positive rate:", round(y.mean(), 4))
print("feature width per day:", feat_dict[ids[0]].shape[1])

admissions: 11022 | positive rate: 0.1751
feature width per day: 171


**How to read this:** we loaded the raw day-by-day arrays and the matching
labels/patients. **Remember:** each `feat_dict[id]` is still a `(n_days, 171)`
table — variable length. The next step turns these into fixed-size tensors the
GRU can batch.

## 2. Build the lab sequence and the static feature vector

Two inputs now, matching the two branches of the model:

1. **Lab sequence** — the 36 lab columns (indices 94:130, the block that
   varies day to day) across the last `N_DAYS` days, left-padded. This goes
   through the GRU.
2. **Static vector** — features that are constant within a stay, taken once per
   admission: age, gender, ethnicity, sequence_length, plus ICD and MED summaries
   (burden counts and med presence/intensity). This bypasses the GRU.

`N_DAYS = 7` — a short discharge-aligned window (median stay is 11 days).

In [3]:
N_DAYS = 7
LAB = slice(94, 130)   # the 36 lab columns (the only block that varies over days)
DEMO, ICD, MED = slice(0,3), slice(3,94), slice(130,171)

def make_lab_window(m, n_days=N_DAYS):
    """Last n_days of the LAB columns only, left-padded. Returns (window, mask)."""
    seq = m[-n_days:, LAB]
    L = seq.shape[0]
    if L < n_days:
        pad = np.zeros((n_days - L, seq.shape[1]), dtype=np.float32)
        seq = np.vstack([pad, seq])
        mask = np.concatenate([np.zeros(n_days - L), np.ones(L)])
    else:
        mask = np.ones(n_days)
    return seq.astype(np.float32), mask.astype(np.float32)

def make_static(m):
    """One constant-per-admission vector: demo + ICD/MED summaries (taken once)."""
    demo = m[0, DEMO]                              # age, gender, ethnicity (3)
    icd_flags = (m[:, ICD].max(0) > 0).astype(np.float32)   # 91 ever-present flags
    icd_count = np.array([icd_flags.sum()], dtype=np.float32)
    med_flags = (m[:, MED].max(0) > 0).astype(np.float32)   # 41 ever-given flags
    med_count = np.array([med_flags.sum()], dtype=np.float32)
    med_intensity = (m[:, MED].sum(0) / m.shape[0]).astype(np.float32)  # 41 per-day avg
    seq_len = np.array([m.shape[0]], dtype=np.float32)
    return np.concatenate([demo, seq_len, icd_count, icd_flags,
                           med_count, med_flags, med_intensity]).astype(np.float32)

N_LAB = 36
X_lab  = np.zeros((len(ids), N_DAYS, N_LAB), dtype=np.float32)
X_mask = np.zeros((len(ids), N_DAYS), dtype=np.float32)
static_list = []
for k, i in enumerate(ids):
    m = feat_dict[i]
    w, mk = make_lab_window(m)
    X_lab[k], X_mask[k] = w, mk
    static_list.append(make_static(m))
X_static = np.vstack(static_list).astype(np.float32)

print("X_lab shape   :", X_lab.shape, "(admissions, days, lab features)")
print("X_static shape:", X_static.shape, "(admissions, static features)")

X_lab shape   : (11022, 7, 36) (admissions, days, lab features)
X_static shape: (11022, 179) (admissions, static features)


**How to read this:** each admission is now two things — a `7 × 36` lab sequence
for the GRU, and a flat static vector for the bypass branch. **Remember:** only
the labs go through the recurrent layer, because only the labs carry a
day-to-day trajectory; everything constant is handled once in the static branch.

## 3. Find which static features need scaling

The lab sequence is all 0/1 flags, so it needs no scaling. The **static**
vector, though, mixes 0/1 flags with large-range numerics (age, counts,
med-intensity), so we note which static columns to standardize — fit on the
training rows only, inside each fold.

In [4]:
# which STATIC columns are large-range numerics (need scaling) vs 0/1 flags?
static_max = X_static.max(axis=0)
static_numeric_cols = np.where(static_max > 1.5)[0]
print("static features total:", X_static.shape[1])
print("static columns needing scaling:", len(static_numeric_cols))

static features total: 179
static columns needing scaling: 44


**How to read this:** the lab sequence is 0/1 (no scaling); only the static
numerics (age, counts, med-intensity) get standardized. **Remember:** scaling is
fit on training rows inside each fold — the list here just marks which static
columns to touch.

## 4. Define the GRU model

The model: a GRU reads the 7-day sequence and produces a summary (its final
hidden state); that summary goes through a small feed-forward head to a single
readmission probability. Dropout is included to fight overfitting on our small
dataset.

In [5]:
import torch.nn as nn

class GRUStaticReadmission(nn.Module):
    """Two branches: GRU over the lab sequence + a linear branch for static
    features, concatenated into a shared head."""
    def __init__(self, n_lab, n_static, hidden=64, static_hidden=32, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=n_lab, hidden_size=hidden, batch_first=True)
        self.static_net = nn.Sequential(
            nn.Linear(n_static, static_hidden), nn.ReLU(), nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + static_hidden, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x_lab, x_static):
        _, h = self.gru(x_lab)          # h: (1, batch, hidden)
        seq_repr = h[-1]                # (batch, hidden) — discharge-day state
        stat_repr = self.static_net(x_static)
        both = torch.cat([seq_repr, stat_repr], dim=1)
        return self.head(both).squeeze(-1)

print("model defined. parameter count:",
      sum(p.numel() for p in GRUStaticReadmission(N_LAB, X_static.shape[1]).parameters()))

model defined. parameter count: 28481


**How to read this:** two branches meet in a shared head — the GRU turns the lab
sequence into a discharge-state summary, the static branch compresses the
constant features, and the head combines both to predict readmission.
**Remember:** the GRU now sees only the 36 varying labs, so its recurrence is
spent entirely on real temporal signal — this is the fix for the v1 dilution.

## 5. Training + evaluation for one fold

This helper trains the GRU on one fold's training admissions and scores AUROC/
PR-AUC on its validation admissions. It scales numerics (fit on train only), uses
class-imbalance-aware loss weighting, early-stops on validation AUROC, and returns
the best validation scores. Keeping it in one function lets the CV loop stay
short.

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import TensorDataset, DataLoader

def train_one_fold(tr_idx, va_idx, hidden=64, dropout=0.3, lr=1e-3,
                   epochs=40, batch_size=128, patience=6, verbose=False):
    # --- scale static numerics, fit on TRAIN rows only ---
    Xs = X_static.copy()
    if len(static_numeric_cols) > 0:
        tr_vals = Xs[tr_idx][:, static_numeric_cols]
        mu = tr_vals.mean(0); sd = tr_vals.std(0) + 1e-6
        Xs[:, static_numeric_cols] = (Xs[:, static_numeric_cols] - mu) / sd

    # --- tensors (two inputs: lab sequence + static) ---
    Xlab_tr = torch.tensor(X_lab[tr_idx]); Xlab_va = torch.tensor(X_lab[va_idx])
    Xst_tr  = torch.tensor(Xs[tr_idx]);    Xst_va  = torch.tensor(Xs[va_idx])
    ytr = torch.tensor(y[tr_idx]);         yva = torch.tensor(y[va_idx])
    train_dl = DataLoader(TensorDataset(Xlab_tr, Xst_tr, ytr),
                          batch_size=batch_size, shuffle=True)

    model = GRUStaticReadmission(N_LAB, X_static.shape[1], hidden=hidden,
                                 dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    pos_weight = torch.tensor([(ytr == 0).sum() / max((ytr == 1).sum(), 1)]).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_auc, best_pr, wait = 0.0, 0.0, 0
    for ep in range(epochs):
        model.train()
        for xb_lab, xb_st, yb in train_dl:
            xb_lab, xb_st, yb = xb_lab.to(device), xb_st.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb_lab, xb_st), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = torch.sigmoid(model(Xlab_va.to(device), Xst_va.to(device))).cpu().numpy()
        auc = roc_auc_score(yva.numpy(), p); pr = average_precision_score(yva.numpy(), p)
        if auc > best_auc:
            best_auc, best_pr, wait = auc, pr, 0
        else:
            wait += 1
            if wait >= patience:
                break
        if verbose:
            print(f"  epoch {ep:2d}: val AUROC {auc:.4f}")
    return best_auc, best_pr

**How to read this:** one fold's worth of training. Key safeguards: numeric
scaling fit on train only (no leakage), `pos_weight` for the imbalance,
early-stopping on validation AUROC (keeps the best epoch, avoids overfitting).
**Remember:** we report the *best* validation AUROC per fold — consistent with how
we scored Track 1 — so the numbers are directly comparable.

*Note on imbalance:* Track 1 found rebalancing hurt AUROC, but neural nets often
need the loss weighting just to train stably on 18% positives. If results
disappoint, trying `pos_weight=1` (no weighting) is a quick thing to test.

## 6. Run patient-grouped cross-validation

The same `StratifiedGroupKFold` on `subject_id` as Track 1, so no patient leaks
across folds and the AUROC is directly comparable to the 0.8032 bar. Each fold
trains a fresh GRU.

In [7]:
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X_lab, y, groups))
for k, (tr, va) in enumerate(folds):
    assert len(set(groups[tr]) & set(groups[va])) == 0, f"leak in fold {k}"

fold_aucs, fold_prs = [], []
for k, (tr, va) in enumerate(folds):
    auc, pr = train_one_fold(tr, va)
    fold_aucs.append(auc); fold_prs.append(pr)
    print(f"fold {k}: AUROC {auc:.4f}  PR-AUC {pr:.4f}")

fold_aucs, fold_prs = np.array(fold_aucs), np.array(fold_prs)
print(f"\nGRU  AUROC {fold_aucs.mean():.4f} ± {fold_aucs.std():.4f}"
      f" | PR-AUC {fold_prs.mean():.4f} ± {fold_prs.std():.4f}")
print(f"Track-1 bar to beat: 0.8032  |  GRU v1 (all 171 cols): 0.7904")

fold 0: AUROC 0.8190  PR-AUC 0.6499
fold 1: AUROC 0.8076  PR-AUC 0.6526
fold 2: AUROC 0.8145  PR-AUC 0.6557
fold 3: AUROC 0.7909  PR-AUC 0.6135
fold 4: AUROC 0.8095  PR-AUC 0.6604

GRU  AUROC 0.8083 ± 0.0095 | PR-AUC 0.6464 ± 0.0169
Track-1 bar to beat: 0.8032  |  GRU v1 (all 171 cols): 0.7904


**How to read this:** the GRU's cross-validated AUROC, directly comparable to
Track 1's 0.8032. **Remember the honest read:** if it's clearly above 0.8032, lab
trajectory shape carries real signal and the sequence model earned its keep. If
it's below or tied, the flat summaries already captured what the labs offer —
a legitimate, informative result, not a failure.

## 7. If it beats Track 1 — next steps; if not — what we learned

**If the GRU beats 0.8032:**
- Run the multi-seed stability check (same as Track 1) to confirm the gain
  survives different fold partitions.
- Try light tuning: `hidden` (32/64/128), `dropout`, `N_DAYS` (5/7/10),
  `pos_weight` on/off.
- Consider an ensemble of the GRU and CatBoost (they may capture different signal).

**If it doesn't beat Track 1:**
- That's evidence the lab trajectory *shape* adds little over the flat summaries —
  a real finding. Track 1's CatBoost stays the chosen model.
- Cheap things worth one try before giving up: a longer window (`N_DAYS=10`),
  bidirectional GRU, or feeding only the lab columns + static context (less noise
  from the constant blocks).

**Either way**, the final submission uses whichever model has the higher
validated CV AUROC, retrained on all labeled data, to predict the project's
`test.csv`.